In [0]:
import os
import json
import urllib.parse
import urllib.request
from delta.tables import DeltaTable
from datetime import date, timedelta
from pyspark.sql import functions as F

In [0]:
catalogo = "databricks_cata_managed"
volume_landing = "cambio_ptax_raw_files"

data_fim = date.today() 
data_inicio = data_fim 

data_inicio_api = data_inicio.strftime("%m-%d-%Y")
data_fim_api = data_fim.strftime("%m-%d-%Y")

data_inicio_ref = data_inicio.strftime("%Y-%m-%d")
data_fim_ref = data_fim.strftime("%Y-%m-%d")

batch_id = f"{data_inicio_ref}_{data_fim_ref}".replace("-", "")

tabela_bronze = f"{catalogo}.bronze.cambio_ptax_raw"
landing_dir = f"/Volumes/{catalogo}/landing/{volume_landing}/batch_{batch_id}"

tabela_bronze = f"{catalogo}.bronze.cambio_ptax_raw"

print(f"Lendo landing: {landing_dir}")
print(f"Salvando bronze: {tabela_bronze}")


In [0]:
df_bronze = (
    spark.read
    .option("multiLine", "true")
    .json(f"{landing_dir}/*.json")
    .withColumn("_arquivo_lido", F.col("_metadata.file_path"))
    .withColumn("_data_ingestao", F.current_timestamp())
    .withColumn("_batch_processamento", F.lit(batch_id))
    .withColumn("_camada", F.lit("bronze"))
    .withColumn("_origem", F.lit("API PTAX - Banco Central"))
)

In [0]:

if not spark.catalog.tableExists(tabela_bronze):
    (
        df_bronze.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(tabela_bronze)
    )

else:
    delta_bronze = DeltaTable.forName(spark, tabela_bronze)

    schema_registros_destino = (
        spark.table(tabela_bronze)
        .schema["registros"]
        .dataType
    )

    df_bronze_ajustado = df_bronze.withColumn(
        "registros",
        F.from_json(
            F.to_json(F.col("registros")),
            schema_registros_destino
        )
    )

    (
        delta_bronze.alias("t")
        .merge(
            df_bronze_ajustado.alias("s"),
            """
            t.batch_id = s.batch_id
            AND t.moeda = s.moeda
            """
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )